# 01 — Temporal evaluation, right-censored labels, and honest metrics

This studybook turns the graph from `00_graph_construction.ipynb` into an evaluation cohort. It assumes only that you know what the instrument and company nodes represent; the statistical ideas are introduced here.

By the end, you should be able to explain:

1. why a random train/test split is inappropriate for transactions over time;
2. what **right censoring** means for an eventual impairment label;
3. why an open positive is known but an open negative is not;
4. how seen-company and cold-start test cases differ;
5. why training and inference require different edge visibility;
6. why PR-AUC is the headline metric and ROC AUC is secondary;
7. how precision@k and recall@k express a finite review budget.

## 1. A prediction problem has a timeline

The table spans several years. A model intended for future invoices must be trained on earlier instruments and assessed on later ones. The v1 cutoff is $T=2018\text{-}04\text{-}30$, matching the original impairment experiment, and outcomes are observed through analysis date $A=2018\text{-}12\text{-}18$.

```mermaid
timeline
    title Evaluation timeline
    2013-07-23 : Historical instruments begin
    2018-04-30 : Cutoff T — train period ends, test period begins
    2018-12-18 : Analysis date A — outcome observation ends
```

A shuffled split would let later economic conditions, companies, and network structure enter training while earlier rows appear in testing. That answers an easier question—interpolation across a mixed historical table—not the deployment question of generalizing forward in time.

## 2. Labels can be unknown, not merely negative

`has_impairment1` is an eventual event label. At the analysis date, an instrument still marked open may impair later. Calling every currently non-impaired open instrument a negative would silently turn unknown futures into known outcomes. This is **right censoring**: observation stopped before the final event/no-event outcome was established.

The target-aware v1 maturity rule is:

$$m_i = \mathbf{1}[d_i \le A] \land (y_i = 1 \lor open_i = 0),$$

where $d_i$ is invoice date, $A$ is the analysis date, and $y_i$ is impairment.

| Snapshot state | Is impairment label known? | Use? |
|---|---:|---:|
| Impaired, closed | Yes—event observed | Yes |
| Impaired, still open | Yes—event already observed | Yes |
| Not impaired, closed | Yes—resolved without event | Yes |
| Not impaired, still open | No—may impair later | No, censored |

This is more precise than dropping every open instrument: seven open rows in the full dataset already have a recorded impairment and therefore have known positive labels.

The mature temporal cohorts are therefore

$$Train = \{i: d_i < T \land m_i=1\},$$

$$Test = \{i: T \le d_i \le A \land m_i=1\}.$$

The graph may retain censored pre-cutoff instruments as **unlabelled structural context**: their existence and origination attributes were known, even though their targets are not used in the loss. A mask controls which instrument labels are supervised.

## 3. Seen versus cold-start test instruments

A company is **seen** if it appears on either side of any pre-cutoff instrument. A test instrument is cold-start if **either** its seller or buyer was absent before the cutoff.

This is deliberately an instrument-level definition: one familiar endpoint does not erase the uncertainty introduced by a brand-new counterparty. Seen and cold-start masks must be disjoint and together cover the entire mature test cohort.

### Transductive versus inductive learning

A **transductive** graph task learns about a fixed set of nodes already present during training; the labels may be hidden, but the entities and topology are known. An **inductive** task must generalize to newly arriving nodes or graphs.

This project is inductive at two levels: post-cutoff invoice nodes are new prediction cases, and many connect to companies never observed before the cutoff. That is why company-ID embeddings would fail for cold-start cases and why company features use observable history with an all-zero fallback.

## 4. Build the graph and evaluation split

The reusable implementation lives in `src/graph_ml/evaluation/`. As before, committed outputs contain aggregate counts only.

In [1]:
from pathlib import Path

import pandas as pd

from graph_ml.data import GraphBuildConfig, build_trade_finance_graph
from graph_ml.evaluation import (
    TemporalSplitConfig,
    build_temporal_evaluation_split,
    build_temporal_graph_views,
    compute_binary_metrics,
)

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists()
)
data_path = repo_root / "data/02_instrumentsdf_2.parquet"
if not data_path.exists():
    raise FileNotFoundError(
        "The real local Parquet data is required; see wiki/this-project/data-availability.md"
    )

instruments = pd.read_parquet(data_path)
graph_result = build_trade_finance_graph(
    instruments, GraphBuildConfig(cutoff="2018-04-30")
)
split = build_temporal_evaluation_split(
    instruments,
    graph_result,
    TemporalSplitConfig(analysis_date="2018-12-18"),
)
split.summary()

{'cutoff': '2018-04-30',
 'analysis_date': '2018-12-18',
 'pre_cutoff_instruments': 46102,
 'post_cutoff_instruments': 13718,
 'mature_train_instruments': 42321,
 'mature_test_instruments': 9293,
 'censored_open_negatives': 8206,
 'seen_test_instruments': 7085,
 'cold_start_test_instruments': 2208,
 'cold_start_test_rate': 0.23759819218766814}

The 8,206 censored rows are specifically **open negatives**, not all open instruments. The maturity filter removes them from both training loss and reported test metrics. It does not delete their observable pre-cutoff topology from the training graph.

In [2]:
labels = graph_result.graph["instrument"].y
cohort_rows = []
for cohort, mask in {
    "train": split.train_mask,
    "test_all": split.test_mask,
    "test_seen": split.seen_test_mask,
    "test_cold_start": split.cold_start_test_mask,
}.items():
    count = int(mask.sum())
    positives = int(labels[mask].sum())
    cohort_rows.append(
        {
            "cohort": cohort,
            "instruments": count,
            "impairments": positives,
            "prevalence": positives / count,
        }
    )
pd.DataFrame(cohort_rows).set_index("cohort").style.format(
    {"prevalence": "{:.2%}"}
)

,instruments,impairments,prevalence
cohort,,,
train,42321,710,1.68%
test_all,9293,522,5.62%
test_seen,7085,292,4.12%
test_cold_start,2208,230,10.42%


The test period is not distributed like the training period: impairment prevalence rises from about 1.68% to 5.62%. Cold-start instruments are especially difficult, with about 10.42% impairment versus 4.12% among seen-company instruments. This is **distribution shift**, and it demonstrates why one aggregate score could conceal operationally important behavior.

## 5. A mask alone does not stop graph leakage

In ordinary tabular ML, a train mask is enough because rows do not exchange information. In a GNN, edges let nodes affect one another. If every post-cutoff instrument can send a message into its company, one test invoice can alter the representation used to score another test invoice.

The implementation therefore creates two node-induced views and returns index mappings back to the full graph:

```mermaid
flowchart LR
    subgraph Training view
      P1[pre-T instruments] --> C1[companies]
      C1 --> P1
    end
    subgraph Inference view
      P2[pre-T instruments] --> C2[companies]
      C2 --> P2
      C2 --> Q2[post-T instruments]
      Q2 -. blocked .-> C2
    end
```

The training view physically excludes post-cutoff nodes, so even global operations such as batch normalization cannot observe them. At inference, post-cutoff instruments can **receive** historical company context but cannot send messages into company states. This produces a conservative fixed-origin evaluation: test cases do not inform one another, even when they share a company.

In [3]:
views = build_temporal_graph_views(graph_result.graph, split)

print(
    "Training view nodes:",
    views.training["instrument"].num_nodes,
    "instruments /",
    views.training["company"].num_nodes,
    "companies",
)
print(
    "Inference view nodes:",
    views.inference["instrument"].num_nodes,
    "instruments /",
    views.inference["company"].num_nodes,
    "companies",
)

edge_rows = []
for edge_type in graph_result.graph.edge_types:
    edge_rows.append(
        {
            "relation": " → ".join(edge_type),
            "full_graph": graph_result.graph[edge_type].num_edges,
            "training_view": views.training[edge_type].num_edges,
            "inference_view": views.inference[edge_type].num_edges,
        }
    )
pd.DataFrame(edge_rows).set_index("relation")

Training view nodes: 46102 instruments / 2500 companies
Inference view nodes: 59820 instruments / 3349 companies


,full_graph,training_view,inference_view
relation,,,
instrument → sold_by → company,59820,46102,46102
company → sells → instrument,59820,46102,59820
instrument → owed_by → company,59820,46102,46102
company → owes → instrument,59820,46102,59820


## 6. Metrics for a rare event

A scoring model ranks instruments by estimated risk. Two familiar curves summarize rankings across thresholds:

- **ROC** plots true-positive rate against false-positive rate. ROC AUC asks how often a random positive ranks above a random negative. It is useful for comparison with the thesis, but thousands of easy true negatives can make performance look strong in an imbalanced problem.
- **Precision–recall** plots precision $TP/(TP+FP)$ against recall $TP/(TP+FN)$. Average precision (reported here as PR-AUC) concentrates directly on retrieving rare positives. Its no-skill reference is approximately the positive prevalence, so it must always be read beside the cohort base rate.

Neither metric chooses an operational threshold. If analysts can investigate only $k$ invoices, then:

$$Precision@k = \frac{\text{impaired instruments in top }k}{k},$$

$$Recall@k = \frac{\text{impaired instruments in top }k}{\text{all impaired instruments}}.$$

### A four-instrument example

The scores below rank one positive first, a negative second, the other positive third, and the final negative last. With a review budget of two instruments, one of the two selected cases is truly impaired.

In [4]:
toy_metrics = compute_binary_metrics(
    y_true=[0, 1, 0, 1],
    y_score=[0.1, 0.9, 0.8, 0.7],
    top_k=2,
)
toy_metrics.as_dict()

{'sample_count': 4,
 'positive_count': 2,
 'prevalence': 0.5,
 'pr_auc': 0.8333333333333333,
 'roc_auc': 0.75,
 'top_k': 2,
 'precision_at_k': 0.5,
 'recall_at_k': 0.5}

The example has PR-AUC $5/6 \approx 0.833$, ROC AUC $0.75$, precision@2 $0.5$, and recall@2 $0.5$. The implementation returns undefined quantities as `None` rather than inventing a number—for example, ROC AUC is undefined when a subgroup contains only one class. This matters when reporting small cold-start slices.

## 7. What is fixed now, and what remains

This step fixes the evaluation contract shared by every baseline and GNN:

- cutoff: 2018-04-30;
- analysis date: 2018-12-18;
- impairment maturity: positive event observed **or** instrument closed;
- open negatives excluded as right-censored;
- PR-AUC primary, ROC AUC secondary;
- precision@k and recall@k reported at a stated review budget;
- overall, seen-company, and cold-start test results reported separately;
- post-cutoff instruments cannot update company representations.

This is a fixed-origin static evaluation, not a rolling deployment simulation. A later temporal phase can allow each test-time instrument to become history for genuinely later predictions, but only with time-ordered message passing. The next notebook will build trivial, logistic-regression, and LightGBM baselines on this exact cohort.

## Takeaways

- A temporal cutoff tests forward generalization; a shuffled split answers an easier and less useful question.
- An unobserved event is not automatically a negative: open non-impaired instruments are right-censored.
- Positive events remain mature even when the instrument is still open.
- Cold-start is defined from pre-cutoff topology and materially changes both cohort size and risk.
- In graph ML, leakage control must restrict message-passing edges as well as label masks.
- PR-AUC is the rare-event headline, ROC AUC preserves historical comparability, and top-k metrics describe an operational review budget.
- Every subsequent model must use these same masks, graph views, and reporting groups.